# EDI Adaptive SSIM Repair Reproduction Audit v1
Run all cells in a CPU Colab runtime. Requires the completed remaining-method v1 pair-plan and integration outputs already on Drive. Recomputes metrics from existing Eigen-CAM and Score-CAM maps only. No training, attribution generation, canonical-table replacement, or figure replacement.

Historical code is extracted verbatim from cell 24 of Untitled38 (2).ipynb. Numerical agreement tests a protocol hypothesis; it does not certify map/model identity or justify replacing a configuration-mismatched baseline. Missing or ambiguous intended pairs remain unresolved. Fingerprints identify files read in this run, not their historical contents.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, os, sys, platform, zipfile
from datetime import datetime, timezone
from functools import lru_cache
import numpy as np
import pandas as pd
import skimage
from skimage.metrics import structural_similarity
from skimage.transform import resize
from IPython.display import display
ROOT=Path('/content/drive/MyDrive/Dissertation')
RUN='edi_adaptive_ssim_repair_reproduction_v1'
SOURCE=ROOT/'provenance_inventory/edi_internal_attribution_provenance_integration_v1/tables/internal_manifest_provenance_corrected.csv'
PLAN=ROOT/'provenance_inventory/edi_remaining_method_protocol_pair_audit_v1/tables/table_refreshed_pair_plan.csv'
OUT=ROOT/'provenance_inventory'/RUN
OUT.mkdir(parents=True,exist_ok=True)
TOL=1e-4
EXPECTED_SOURCE_SHA='5714ca0dde44b05fe35df1e30059f93704605efe31e3b7a5f16c44b2bc8d295d'
K=['experiment_id','image_id','preprocess','resolution','attribution_method']
def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def write_csv(name,d):
 p=OUT/name;t=p.with_suffix('.tmp');d.to_csv(t,index=False);os.replace(t,p)
def write_json(name,d):
 p=OUT/name;t=p.with_suffix('.tmp');t.write_text(json.dumps(d,indent=2,allow_nan=False));os.replace(t,p)
source_sha=sha(SOURCE);plan_sha=sha(PLAN)
assert source_sha==EXPECTED_SOURCE_SHA, 'Integration manifest differs from reviewed version; stop and reconcile.'
a=pd.read_csv(SOURCE,low_memory=False);p=pd.read_csv(PLAN,low_memory=False)
for d in (a,p):
 d.image_id=d.image_id.astype(str).str.upper()
 d.resolution=pd.to_numeric(d.resolution).astype(int)
assert not a.duplicated(K).any() and not p.duplicated(K).any()
a=a[a.attribution_method.isin(['eigencam','scorecam'])].copy()
assert len(a)==7680
rows=a.merge(p,on=K,validate='one_to_one',suffixes=('','_plan'))
assert len(rows)==7680
print('Rows to inspect:',len(rows))


In [ ]:
def normalize_heatmap_for_compare(h):
    h = np.asarray(h, dtype=np.float32)
    h = np.squeeze(h)
    h = np.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)

    h_min = np.min(h)
    h_max = np.max(h)

    if h_max - h_min > 0:
        h = (h - h_min) / (h_max - h_min)
    else:
        h = np.zeros_like(h, dtype=np.float32)

    return h.astype(np.float32)

def get_safe_ssim_win_size(shape):
    min_dim = int(min(shape))

    if min_dim >= 7:
        return 7
    elif min_dim >= 5:
        return 5
    elif min_dim >= 3:
        return 3
    else:
        return None

def compute_heatmap_pair_metrics(base_hm, exp_hm):
    base_hm = normalize_heatmap_for_compare(base_hm)
    exp_hm = normalize_heatmap_for_compare(exp_hm)

    if base_hm.shape != exp_hm.shape:
        exp_hm = resize(
            exp_hm,
            base_hm.shape,
            preserve_range=True,
            anti_aliasing=True
        ).astype(np.float32)

    b = base_hm.flatten()
    e = exp_hm.flatten()

    if np.std(b) == 0 or np.std(e) == 0:
        corr = 0.0
    else:
        corr = float(np.corrcoef(b, e)[0, 1])

    corr = float(np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0))

    win_size = get_safe_ssim_win_size(base_hm.shape)

    if win_size is None:
        base_for_ssim = resize(base_hm, (8, 8), preserve_range=True, anti_aliasing=True).astype(np.float32)
        exp_for_ssim = resize(exp_hm, (8, 8), preserve_range=True, anti_aliasing=True).astype(np.float32)

        ssim_val = float(
            structural_similarity(
                base_for_ssim,
                exp_for_ssim,
                data_range=1.0,
                win_size=7
            )
        )
    else:
        ssim_val = float(
            structural_similarity(
                base_hm,
                exp_hm,
                data_range=1.0,
                win_size=win_size
            )
        )

    ssim_val = float(np.nan_to_num(ssim_val, nan=0.0, posinf=0.0, neginf=0.0))

    corr_drift = 1.0 - ((corr + 1.0) / 2.0)
    ssim_drift = 1.0 - ssim_val
    edi_val = (corr_drift + ssim_drift) / 2.0

    return corr, ssim_val, corr_drift, ssim_drift, edi_val

In [ ]:
write_json("historical_protocol.json", {'source_notebook': 'Untitled38 (2).ipynb', 'cell_index_zero_based': 24, 'source_cell_sha256': 'fa379a4ec2378632d147c3abceb1f4f237c7114bb0535eebfbe3ef57fbd2b55d', 'extracted_functions_sha256': '3aec0b426b00d4a7d4a1a080da0d9709ad4f6c4e5bc1c2212672b686ea95e1d9', 'extracted_functions': 'def normalize_heatmap_for_compare(h):\n    h = np.asarray(h, dtype=np.float32)\n    h = np.squeeze(h)\n    h = np.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)\n\n    h_min = np.min(h)\n    h_max = np.max(h)\n\n    if h_max - h_min > 0:\n        h = (h - h_min) / (h_max - h_min)\n    else:\n        h = np.zeros_like(h, dtype=np.float32)\n\n    return h.astype(np.float32)\n\ndef get_safe_ssim_win_size(shape):\n    min_dim = int(min(shape))\n\n    if min_dim >= 7:\n        return 7\n    elif min_dim >= 5:\n        return 5\n    elif min_dim >= 3:\n        return 3\n    else:\n        return None\n\ndef compute_heatmap_pair_metrics(base_hm, exp_hm):\n    base_hm = normalize_heatmap_for_compare(base_hm)\n    exp_hm = normalize_heatmap_for_compare(exp_hm)\n\n    if base_hm.shape != exp_hm.shape:\n        exp_hm = resize(\n            exp_hm,\n            base_hm.shape,\n            preserve_range=True,\n            anti_aliasing=True\n        ).astype(np.float32)\n\n    b = base_hm.flatten()\n    e = exp_hm.flatten()\n\n    if np.std(b) == 0 or np.std(e) == 0:\n        corr = 0.0\n    else:\n        corr = float(np.corrcoef(b, e)[0, 1])\n\n    corr = float(np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0))\n\n    win_size = get_safe_ssim_win_size(base_hm.shape)\n\n    if win_size is None:\n        base_for_ssim = resize(base_hm, (8, 8), preserve_range=True, anti_aliasing=True).astype(np.float32)\n        exp_for_ssim = resize(exp_hm, (8, 8), preserve_range=True, anti_aliasing=True).astype(np.float32)\n\n        ssim_val = float(\n            structural_similarity(\n                base_for_ssim,\n                exp_for_ssim,\n                data_range=1.0,\n                win_size=7\n            )\n        )\n    else:\n        ssim_val = float(\n            structural_similarity(\n                base_hm,\n                exp_hm,\n                data_range=1.0,\n                win_size=win_size\n            )\n        )\n\n    ssim_val = float(np.nan_to_num(ssim_val, nan=0.0, posinf=0.0, neginf=0.0))\n\n    corr_drift = 1.0 - ((corr + 1.0) / 2.0)\n    ssim_drift = 1.0 - ssim_val\n    edi_val = (corr_drift + ssim_drift) / 2.0\n\n    return corr, ssim_val, corr_drift, ssim_drift, edi_val'})


In [ ]:
# Only the three historical metric helpers above are executed; repair-writing code is excluded.
# Small checks exercise adaptive windows and the historical constant-map convention.
for size,win in [(4,3),(5,5),(6,5),(7,7)]:
 x=np.arange(size*size,dtype=np.float32).reshape(size,size)
 assert get_safe_ssim_win_size(x.shape)==win
 result=compute_heatmap_pair_metrics(x,x)
 assert np.allclose([result[0],result[1],result[4]],[1,1,0],atol=1e-6)
assert compute_heatmap_pair_metrics(np.zeros((4,4)),np.zeros((4,4)))[0]==0
print('Historical protocol checks passed.')


In [ ]:
fingerprints={}
@lru_cache(maxsize=256)
def load_map(path):
 path=str(path);arr=np.load(path,allow_pickle=False)
 fingerprints[path]={'path':path,'sha256':sha(path),'shape':json.dumps(list(arr.shape)),
 'nonfinite_count':int((~np.isfinite(arr)).sum()),'constant_after_normalization':bool(np.ptp(normalize_heatmap_for_compare(arr))==0)}
 return arr
results=[]
for number,(_,r) in enumerate(rows.iterrows(),1):
 comparison=json.loads(r.comparison_candidates)
 intended=json.loads(r.baseline_candidates)
 image=str(r.get('source_image_id_plan',r.image_id))
 recorded=ROOT/'edi_exports/heatmaps'/r.attribution_method/str(r.recorded_baseline_id)/f'{image}_{r.attribution_method}_heatmap.npy'
 pairs=[('recorded', [str(recorded)]),('intended',intended)]
 for pairing,bases in pairs:
  out={k:r[k] for k in K};out.update(pairing=pairing,recorded_baseline_id=str(r.recorded_baseline_id),
   intended_baseline_id=str(r.intended_baseline_id),baseline_path='',comparison_path='',status='unresolved',
   numerical_match=False,map_model_provenance_verified=False)
  if len(comparison)!=1 or len(bases)!=1:
   out['status']='missing_or_ambiguous_candidate'
  else:
   bp,ep=str(bases[0]),str(comparison[0]);out.update(baseline_path=bp,comparison_path=ep)
   if not Path(bp).is_file() or not Path(ep).is_file():out['status']='missing_file'
   else:
    try:
     b,e=load_map(bp),load_map(ep)
     values=compute_heatmap_pair_metrics(b,e)
     out.update(status='calculated',ssim_win_size=get_safe_ssim_win_size(np.squeeze(b).shape))
     errors=[]
     for name,value,target in zip(['pearson_corr','ssim','edi'],[values[0],values[1],values[4]],[r.pearson_corr,r.ssim,r.edi_primary]):
      err=abs(float(value)-float(target));out['recomputed_'+name]=float(value);out[name+'_absolute_error']=err;errors.append(err)
     out['numerical_match']=bool(all(np.isfinite(x) and x<=TOL for x in errors))
     out['baseline_constant']=fingerprints[bp]['constant_after_normalization']
     out['comparison_constant']=fingerprints[ep]['constant_after_normalization']
    except Exception as exc:out.update(status='calculation_error',error=f'{type(exc).__name__}: {exc}')
  results.append(out)
 if number%250==0 or number==len(rows):
  write_csv('pair_reproduction.csv',pd.DataFrame(results))
  print(f'{number}/{len(rows)} rows inspected',flush=True)
result=pd.DataFrame(results)
summary=result.groupby(['attribution_method','pairing','status'],dropna=False).agg(rows=('image_id','size'),numerical_matches=('numerical_match','sum')).reset_index()
write_csv('summary.csv',summary)
write_csv('map_fingerprints.csv',pd.DataFrame(list(fingerprints.values())))
assert sha(SOURCE)==source_sha and sha(PLAN)==plan_sha,'Input changed during audit'
display(summary)


In [ ]:
status={'run_complete':True,'audit_scope':'Eigen-CAM and Score-CAM adaptive SSIM hypothesis',
 'source_sha256':source_sha,'pair_plan_sha256':plan_sha,'comparison_rows':len(rows),'pair_attempts':len(result),
 'tolerance':TOL,'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,'skimage':skimage.__version__,
 'completed_utc':datetime.now(timezone.utc).isoformat(),'canonical_sources_unchanged':True,
 'attributions_generated':False,'models_retrained':False,'full_provenance_audit_complete':False,
 'interpretation':'Numerical matches do not prove historical execution or map/model provenance. Recorded mismatched baselines remain unsuitable for configuration-correct conclusions.'}
write_json('final_status.json',status)
archive=OUT.parent/(RUN+'_compact_bundle.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for name in ['pair_reproduction.csv','summary.csv','map_fingerprints.csv','final_status.json','historical_protocol.json']:
  z.write(OUT/name,arcname=name)
print('Upload this completed bundle for review:',archive)
from google.colab import files
files.download(str(archive))
